# Spectral Superposition Analysis

Comprehensive analysis of superposition dynamics through the lens of association schemes and spectral decomposition.

## Theoretical Framework

For the Gram matrix $M = W^\intercal W$, the **spectral ansatz** proposes:
$$M \approx \mu E_o$$
where $E_o$ is the primitive idempotent corresponding to a particular geometry.

This leads to:
$$D_i = \frac{(M_{ii})^2}{(M^2)_{ii}} = \frac{\|W_i\|^2}{\mu}$$

The **resolvent/Stieltjes transform** for block-diagonal decomposition:
$$m(z) = \frac{1}{n}\text{Tr}(M-zI)^{-1} = \sum_{j=1}^k\frac{m_j/n}{\mu_j-z} - \frac{n-\sum_jm_j}{nz}$$

Key observables:
- Number of poles = number of distinct geometries
- Residues = multiplicities $m_j$

In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap
from mpl_toolkits.mplot3d import Axes3D
from pathlib import Path
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

DATA_DIR = Path('../start')
OUTPUT_DIR = Path('.')
N_FEATURES = 1024

## 1. Data Loading and Validation

In [ ]:
def parse_filename(fname):
    """Parse experiment parameters from filename."""
    # Format: n{n}_m{m}_s{sparsity}_seed{seed}.h5
    parts = fname.stem.split('_')
    return {
        'n': int(parts[0][1:]),
        'm': int(parts[1][1:]),
        's': float(parts[2][1:]),
        'seed': int(parts[3][4:])
    }

def load_experiment(fpath):
    """Load a single experiment file with validation."""
    try:
        with h5py.File(fpath, 'r') as f:
            data = {
                'n_features': f.attrs['n_features'],
                'm_hidden': f.attrs['m_hidden'],
                'sparsity': f.attrs['sparsity'],
                'seed': f.attrs['seed'],
                'checkpoint_steps': f['checkpoint_steps'][:],
                'weights': f['weights'][:],
                'fractional_dims': f['fractional_dims'][:],
                'feature_norms': f['feature_norms'][:],
                'biases': f['biases'][:],
                'losses': f['losses'][:]
            }
            
            # Validation checks
            valid = True
            issues = []
            
            # Check for NaN/Inf
            for key in ['fractional_dims', 'feature_norms', 'weights', 'losses']:
                if np.any(np.isnan(data[key])):
                    issues.append(f'NaN in {key}')
                    valid = False
                if np.any(np.isinf(data[key])):
                    issues.append(f'Inf in {key}')
                    valid = False
            
            # Check expected shapes
            n_checkpoints = len(data['checkpoint_steps'])
            if data['fractional_dims'].shape != (n_checkpoints, data['n_features']):
                issues.append('fractional_dims shape mismatch')
                valid = False
            
            data['valid'] = valid
            data['issues'] = issues
            return data
            
    except Exception as e:
        return {'valid': False, 'issues': [str(e)]}

# Discover all files
files = sorted(DATA_DIR.glob('*.h5'))
print(f"Found {len(files)} experiment files")

# Parse metadata
experiments = []
for f in files:
    params = parse_filename(f)
    params['path'] = f
    experiments.append(params)

# Get unique parameter values
m_values = sorted(set(e['m'] for e in experiments))
s_values = sorted(set(e['s'] for e in experiments))
seeds = sorted(set(e['seed'] for e in experiments))

print(f"m_hidden values: {len(m_values)} unique ({min(m_values)} to {max(m_values)})")
print(f"sparsity values: {len(s_values)} unique ({min(s_values):.3f} to {max(s_values):.3f})")
print(f"seeds: {seeds}")
print(f"Expected total: {len(m_values) * len(s_values) * len(seeds)} = {len(m_values)}x{len(s_values)}x{len(seeds)}")

## 2. Completion Heatmap

In [ ]:
def create_completion_heatmap(experiments, m_values, s_values, seeds):
    """Create heatmap showing which experiments are present."""
    
    # Create lookup
    experiment_lookup = {}
    for e in experiments:
        key = (e['m'], round(e['s'], 6), e['seed'])
        experiment_lookup[key] = e['path']
    
    fig, axes = plt.subplots(1, len(seeds), figsize=(8*len(seeds), 10))
    if len(seeds) == 1:
        axes = [axes]
    
    for ax, seed in zip(axes, seeds):
        completion = np.zeros((len(m_values), len(s_values)))
        
        for i, m in enumerate(m_values):
            for j, s in enumerate(s_values):
                key = (m, round(s, 6), seed)
                completion[i, j] = 1 if key in experiment_lookup else 0
        
        im = ax.imshow(completion, aspect='auto', cmap='RdYlGn', 
                       vmin=0, vmax=1, origin='lower')
        ax.set_xlabel('Sparsity Index')
        ax.set_ylabel('m_hidden Index')
        ax.set_title(f'Seed {seed}: {int(completion.sum())}/{completion.size} complete')
        
        # Add axis ticks
        ax.set_xticks(np.arange(0, len(s_values), 10))
        ax.set_xticklabels([f'{s_values[i]:.2f}' for i in range(0, len(s_values), 10)], rotation=45)
        ax.set_yticks(np.arange(0, len(m_values), 5))
        ax.set_yticklabels([str(m_values[i]) for i in range(0, len(m_values), 5)])
    
    plt.suptitle('Experiment Completion Status', fontsize=14, weight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'completion_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Summary stats
    total_expected = len(m_values) * len(s_values) * len(seeds)
    total_found = len(experiments)
    print(f"\nCompletion: {total_found}/{total_expected} ({100*total_found/total_expected:.1f}%)")

create_completion_heatmap(experiments, m_values, s_values, seeds)

## 3. Validate and Load All Data

In [ ]:
# Load all data with validation
all_data = {}
corrupted = []

for exp in tqdm(experiments, desc='Loading experiments'):
    data = load_experiment(exp['path'])
    key = (exp['m'], exp['s'], exp['seed'])
    
    if data['valid']:
        all_data[key] = data
    else:
        corrupted.append({'params': exp, 'issues': data['issues']})

print(f"\nLoaded {len(all_data)} valid experiments")
if corrupted:
    print(f"Found {len(corrupted)} corrupted files:")
    for c in corrupted[:5]:  # Show first 5
        print(f"  {c['params']['path'].name}: {c['issues']}")

## 4. D_i Histograms by Sparsity and Capacity Bins

In [ ]:
def plot_di_histograms(all_data, n_sparsity_bins=5, n_capacity_bins=4):
    """Plot D_i histograms for different bins of sparsity and model capacity."""
    
    # Collect final-checkpoint D_i values
    s_vals = np.array([k[1] for k in all_data.keys()])
    m_vals = np.array([k[0] for k in all_data.keys()])
    rho_vals = N_FEATURES / m_vals  # compression ratio
    
    s_bins = np.linspace(0, 1, n_sparsity_bins + 1)
    rho_bins = np.linspace(rho_vals.min(), rho_vals.max(), n_capacity_bins + 1)
    
    fig, axes = plt.subplots(n_capacity_bins, n_sparsity_bins, figsize=(4*n_sparsity_bins, 3*n_capacity_bins))
    
    for i in range(n_capacity_bins):
        for j in range(n_sparsity_bins):
            ax = axes[i, j]
            
            # Collect D_i values for this bin
            di_values = []
            for (m, s, seed), data in all_data.items():
                rho = N_FEATURES / m
                if rho_bins[i] <= rho < rho_bins[i+1] and s_bins[j] <= s < s_bins[j+1]:
                    # Final checkpoint D_i
                    di_values.extend(data['fractional_dims'][-1].tolist())
            
            if di_values:
                ax.hist(di_values, bins=50, density=True, alpha=0.7, color='steelblue', edgecolor='white')
                ax.set_xlim(0, 1.1)
                ax.axvline(np.mean(di_values), color='red', linestyle='--', label=f'mu={np.mean(di_values):.3f}')
                ax.legend(fontsize=8)
            
            if i == 0:
                ax.set_title(f'S in [{s_bins[j]:.2f},{s_bins[j+1]:.2f})', fontsize=10)
            if j == 0:
                ax.set_ylabel(f'rho in [{rho_bins[i]:.1f},{rho_bins[i+1]:.1f})\nDensity', fontsize=10)
            if i == n_capacity_bins - 1:
                ax.set_xlabel('D_i', fontsize=10)
    
    plt.suptitle('Fractional Dimensionality Distribution by Sparsity and Compression Ratio', fontsize=14, weight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'di_histograms_binned.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_di_histograms(all_data)

## 5. 3D Evolution of D_i Histogram During Training

In [ ]:
def plot_3d_histogram_evolution(all_data, m_target=256, s_target=0.5, seed=0):
    """Plot 3D evolution of D_i histogram during training.
    
    X-axis: D_i value (histogram bins)
    Y-axis: Training step
    Z-axis: Histogram count/density
    """
    
    # Find closest matching experiment
    best_key = None
    best_dist = float('inf')
    for (m, s, sd) in all_data.keys():
        if sd == seed:
            dist = abs(m - m_target) / 100 + abs(s - s_target)
            if dist < best_dist:
                best_dist = dist
                best_key = (m, s, sd)
    
    if best_key is None:
        print(f"No matching experiment found for m={m_target}, s={s_target}")
        return
    
    data = all_data[best_key]
    print(f"Using experiment: m={best_key[0]}, s={best_key[1]:.3f}, seed={best_key[2]}")
    
    fractional_dims = data['fractional_dims']  # (n_checkpoints, n_features)
    steps = data['checkpoint_steps']
    
    # Create histogram data for each checkpoint
    n_bins = 50
    bin_edges = np.linspace(0, 1, n_bins + 1)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    
    fig = plt.figure(figsize=(14, 10))
    ax = fig.add_subplot(111, projection='3d')
    
    # Subsample checkpoints for clarity
    checkpoint_indices = np.linspace(0, len(steps)-1, min(20, len(steps))).astype(int)
    
    cmap = plt.get_cmap('viridis')
    
    for idx, cp_idx in enumerate(checkpoint_indices):
        step = steps[cp_idx]
        di = fractional_dims[cp_idx]
        
        hist, _ = np.histogram(di, bins=bin_edges, density=True)
        
        color = cmap(idx / len(checkpoint_indices))
        ax.bar(bin_centers, hist, zs=step, zdir='y', width=0.015, alpha=0.7, color=color)
    
    ax.set_xlabel('Fractional Dimensionality $D_i$')
    ax.set_ylabel('Training Step')
    ax.set_zlabel('Density')
    ax.set_title(f'D_i Distribution Evolution\nm={best_key[0]}, S={best_key[1]:.3f}', fontsize=14, weight='bold')
    
    ax.view_init(elev=25, azim=-60)
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'di_3d_evolution.png', dpi=150, bbox_inches='tight')
    plt.show()

# Plot for a few representative experiments
plot_3d_histogram_evolution(all_data, m_target=256, s_target=0.5)
plot_3d_histogram_evolution(all_data, m_target=128, s_target=0.3)
plot_3d_histogram_evolution(all_data, m_target=400, s_target=0.8)

## 6. Gradient-Colored Grid (Sparsity x Compression)

In [ ]:
def hex_to_gradient_value(hex_color):
    """Convert hex color to gradient value using blue channel."""
    r = int(hex_color[0:2], 16)
    g = int(hex_color[2:4], 16)
    b = int(hex_color[4:6], 16)
    return (240 - b) / 240

def create_di_gradient_grid(all_data, m_values, s_values):
    """Create rectangular grid colored by D_i gradient."""
    
    # Create custom colormap (based on the gradient function)
    # Blue channel varies: high blue = low value, low blue = high value
    colors_list = [(0.0, 0.1, 1.0), (0.0, 0.5, 0.8), (0.2, 0.7, 0.5), 
                   (0.5, 0.8, 0.2), (0.8, 0.9, 0.0), (1.0, 1.0, 0.0)]
    cmap = LinearSegmentedColormap.from_list('di_gradient', colors_list)
    
    # Average D_i across seeds
    compression_ratios = [N_FEATURES / m for m in m_values]
    
    mean_di_grid = np.full((len(m_values), len(s_values)), np.nan)
    
    for i, m in enumerate(m_values):
        for j, s in enumerate(s_values):
            di_vals = []
            for seed in [0, 1]:
                key = (m, s, seed)
                if key in all_data:
                    di_vals.extend(all_data[key]['fractional_dims'][-1].tolist())
            if di_vals:
                mean_di_grid[i, j] = np.mean(di_vals)
    
    fig, ax = plt.subplots(figsize=(16, 10))
    
    im = ax.imshow(mean_di_grid, aspect='auto', cmap=cmap, origin='lower',
                   extent=[min(s_values), max(s_values), 
                           min(compression_ratios), max(compression_ratios)])
    
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Mean Fractional Dimensionality $\\langle D_i \\rangle$', fontsize=12)
    
    ax.set_xlabel('Sparsity $S$', fontsize=12)
    ax.set_ylabel('Compression Ratio $n/m$', fontsize=12)
    ax.set_title('Fractional Dimensionality Landscape', fontsize=14, weight='bold')
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'di_gradient_grid.png', dpi=150, bbox_inches='tight')
    plt.show()

create_di_gradient_grid(all_data, m_values, s_values)

## 6b. Mean and Std D_i Heatmaps

In [ ]:
def plot_di_heatmaps(all_data, m_values, s_values):
    """Plot heatmaps of mean and std D_i."""
    
    mean_grid = np.full((len(m_values), len(s_values)), np.nan)
    std_grid = np.full((len(m_values), len(s_values)), np.nan)
    
    for i, m in enumerate(m_values):
        for j, s in enumerate(s_values):
            di_vals = []
            for seed in [0, 1]:
                key = (m, s, seed)
                if key in all_data:
                    di_vals.extend(all_data[key]['fractional_dims'][-1].tolist())
            if di_vals:
                mean_grid[i, j] = np.mean(di_vals)
                std_grid[i, j] = np.std(di_vals)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))
    
    # Mean heatmap
    im1 = ax1.imshow(mean_grid, aspect='auto', cmap='viridis', origin='lower')
    ax1.set_xlabel('Sparsity Index')
    ax1.set_ylabel('m_hidden Index')
    ax1.set_title('Mean $D_i$', fontsize=14, weight='bold')
    cbar1 = plt.colorbar(im1, ax=ax1)
    cbar1.set_label('$\\langle D_i \\rangle$')
    
    # Std heatmap  
    im2 = ax2.imshow(std_grid, aspect='auto', cmap='plasma', origin='lower')
    ax2.set_xlabel('Sparsity Index')
    ax2.set_ylabel('m_hidden Index')
    ax2.set_title('Std $D_i$', fontsize=14, weight='bold')
    cbar2 = plt.colorbar(im2, ax=ax2)
    cbar2.set_label('$\\sigma(D_i)$')
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'di_mean_std_heatmaps.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_di_heatmaps(all_data, m_values, s_values)

## 7. Spectral Phase Diagram

In [ ]:
def collect_spectral_data(all_data, subsample_prob=0.1):
    """Collect feature statistics for phase diagram."""
    
    all_norms = []
    all_dims = []
    all_rhos = []  # Compression ratio
    all_sparsities = []
    
    for (m, s, seed), data in tqdm(all_data.items(), desc='Collecting data'):
        rho = N_FEATURES / m
        
        final_dims = data['fractional_dims'][-1]
        final_norms = data['feature_norms'][-1]
        
        # Subsample
        if subsample_prob < 1.0:
            mask = np.random.rand(len(final_dims)) < subsample_prob
            final_dims = final_dims[mask]
            final_norms = final_norms[mask]
        
        all_norms.append(final_norms)
        all_dims.append(final_dims)
        all_rhos.append(np.full(len(final_norms), rho))
        all_sparsities.append(np.full(len(final_norms), s))
    
    X = np.concatenate(all_norms)  # ||W_i||^2
    Y = np.concatenate(all_dims)   # D_i
    C = np.concatenate(all_rhos)   # rho = n/m
    S = np.concatenate(all_sparsities)
    
    return X, Y, C, S

def plot_spectral_phase_diagram(X, Y, C):
    """Feature Norm vs Fractional Dimensionality, colored by Compression Ratio."""
    
    print(f"Plotting {len(X):,} features...")
    
    fig, ax = plt.subplots(figsize=(14, 10), dpi=150)
    
    sc = ax.scatter(X, Y, c=C, cmap='turbo', s=2, alpha=0.3, rasterized=True)
    
    cbar = plt.colorbar(sc, ax=ax)
    cbar.set_label('Compression Ratio $n/m$', fontsize=12, weight='bold')
    
    ax.set_xlabel(r'Feature Norm $\|W_i\|^2$', fontsize=12)
    ax.set_ylabel(r'Fractional Dimensionality $D_i$', fontsize=12)
    ax.set_title('Spectral Superposition: Phase Diagram', fontsize=16, weight='bold')
    
    # Reference lines for different mu values (D = ||W||^2 / mu)
    x_ref = np.linspace(0, np.percentile(X, 99), 100)
    for mu in [1, 2, 3, 4, 5]:
        ax.plot(x_ref, x_ref / mu, '--', alpha=0.4, label=f'mu={mu}')
    
    ax.legend(loc='upper right', fontsize=9)
    ax.set_xlim(left=0)
    ax.set_ylim(bottom=0, top=1.1)
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'spectral_phase_diagram.png', dpi=150, bbox_inches='tight')
    plt.show()

X, Y, C, S = collect_spectral_data(all_data, subsample_prob=0.2)
plot_spectral_phase_diagram(X, Y, C)

## 8. Polar Spectral Ray Plot

In [ ]:
def plot_polar_spectral_rays(X, Y, S):
    """Polar coordinates visualization of spectral quantization.
    
    If D_i = ||W_i||^2 / mu, features cluster around angles theta = arctan(1/mu).
    """
    
    # Filter zero-norm features
    mask = X > 1e-6
    X_clean = X[mask]
    Y_clean = Y[mask]
    S_clean = S[mask]
    
    # Polar coordinates
    r = np.sqrt(X_clean)  # ||W_i||
    theta = np.arctan2(Y_clean, X_clean)  # angle in (D_i, ||W||^2) space
    
    fig = plt.figure(figsize=(16, 8))
    
    # Polar projection
    ax1 = fig.add_subplot(121, projection='polar')
    sc1 = ax1.scatter(theta, r, c=S_clean, cmap='viridis', s=2, alpha=0.5, rasterized=True)
    ax1.set_title("Spectral Quantization (Polar)\n$r=\|W_i\|, \\theta=\\arctan(D_i / \|W_i\|^2)$", pad=20)
    ax1.set_thetamin(0)
    ax1.set_thetamax(90)
    
    # Histogram of angles
    ax2 = fig.add_subplot(122)
    ax2.hist(theta, bins=200, color='steelblue', alpha=0.7, density=True, edgecolor='white', linewidth=0.5)
    
    # Mark reference angles for integer mu
    for mu in [1, 2, 3, 4, 5, 6, 8, 10]:
        ref_angle = np.arctan(1/mu)
        ax2.axvline(ref_angle, color='red', linestyle='--', alpha=0.5, label=f'mu={mu}' if mu <= 5 else '')
    
    ax2.set_title("Density of Spectral Angles", fontsize=14, weight='bold')
    ax2.set_xlabel("Angle $\\theta$ (rad)")
    ax2.set_ylabel("Density")
    ax2.legend(loc='upper right')
    
    cbar = fig.colorbar(sc1, ax=[ax1, ax2], fraction=0.02, pad=0.05)
    cbar.set_label('Sparsity $S$', fontsize=12)
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'polar_spectral_rays.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_polar_spectral_rays(X, Y, S)

## 9. Training Evolution Metrics

In [ ]:
def compute_angular_entropy(di, norms, n_bins=50):
    """Compute entropy of the angular distribution."""
    mask = norms > 1e-6
    if mask.sum() < 10:
        return np.nan
    
    theta = np.arctan2(di[mask], norms[mask])
    hist, _ = np.histogram(theta, bins=n_bins, range=(0, np.pi/2), density=True)
    hist = hist[hist > 0]  # Remove zeros for log
    return -np.sum(hist * np.log(hist + 1e-10)) * (np.pi/2 / n_bins)

def compute_reference_concentration(di, norms, mu_refs=[1, 2, 3, 4, 5], tol=0.1):
    """Fraction of features near reference angles theta = arctan(1/mu)."""
    mask = norms > 1e-6
    if mask.sum() < 10:
        return np.nan
    
    theta = np.arctan2(di[mask], norms[mask])
    ref_angles = [np.arctan(1/mu) for mu in mu_refs]
    
    near_ref = np.zeros(len(theta), dtype=bool)
    for ref in ref_angles:
        near_ref |= np.abs(theta - ref) < tol
    
    return near_ref.mean()

def plot_training_evolution(all_data, m_target=256, s_target=0.5, seed=0):
    """Plot evolution of structure metrics during training."""
    
    # Find matching experiment
    best_key = None
    best_dist = float('inf')
    for (m, s, sd) in all_data.keys():
        if sd == seed:
            dist = abs(m - m_target) / 100 + abs(s - s_target)
            if dist < best_dist:
                best_dist = dist
                best_key = (m, s, sd)
    
    if best_key is None:
        print("No matching experiment found")
        return
    
    data = all_data[best_key]
    print(f"Analyzing: m={best_key[0]}, s={best_key[1]:.3f}")
    
    steps = data['checkpoint_steps']
    losses = data['losses']
    frac_dims = data['fractional_dims']
    norms = data['feature_norms']
    
    # Compute metrics at each checkpoint
    angular_entropy = []
    ref_concentration = []
    active_fraction = []  # Features with ||W_i||^2 > 0.01
    
    for i in range(len(steps)):
        angular_entropy.append(compute_angular_entropy(frac_dims[i], norms[i]))
        ref_concentration.append(compute_reference_concentration(frac_dims[i], norms[i]))
        active_fraction.append((norms[i] > 0.01).mean())
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Loss
    axes[0, 0].semilogy(steps, losses, 'b-', linewidth=2)
    axes[0, 0].set_xlabel('Training Step')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Training Loss')
    
    # Angular entropy
    axes[0, 1].plot(steps, angular_entropy, 'g-', linewidth=2)
    axes[0, 1].set_xlabel('Training Step')
    axes[0, 1].set_ylabel('Angular Entropy')
    axes[0, 1].set_title('Angular Distribution Entropy')
    
    # Reference concentration
    axes[1, 0].plot(steps, ref_concentration, 'r-', linewidth=2)
    axes[1, 0].set_xlabel('Training Step')
    axes[1, 0].set_ylabel('Concentration')
    axes[1, 0].set_title('Concentration near Reference Angles')
    
    # Active features
    axes[1, 1].plot(steps, active_fraction, 'purple', linewidth=2)
    axes[1, 1].set_xlabel('Training Step')
    axes[1, 1].set_ylabel('Fraction')
    axes[1, 1].set_title('Fraction of Active Features ($\|W_i\|^2 > 0.01$)')
    
    plt.suptitle(f'Training Dynamics: m={best_key[0]}, S={best_key[1]:.3f}', fontsize=14, weight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'training_evolution.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_training_evolution(all_data, m_target=256, s_target=0.5)
plot_training_evolution(all_data, m_target=128, s_target=0.7)

---

# Spectral Analysis: Association Scheme Framework

## 10. Eigenvalue Analysis of $M = W^\intercal W$

In [ ]:
def compute_gram_spectrum(W):
    """Compute eigenvalues of the Gram matrix M = W^T W."""
    M = W.T @ W
    eigenvalues = np.linalg.eigvalsh(M)  # Real symmetric matrix
    return np.sort(eigenvalues)[::-1]  # Descending order

def compute_power_traces(W, max_k=4):
    """Compute Tr(M^k) for k = 1, 2, ..., max_k."""
    M = W.T @ W
    traces = []
    M_power = M.copy()
    for k in range(1, max_k + 1):
        traces.append(np.trace(M_power))
        if k < max_k:
            M_power = M_power @ M
    return traces

def analyze_experiment_spectrum(data):
    """Full spectral analysis of an experiment."""
    
    # Get final weights
    W = data['weights'][-1]  # (m, n)
    
    # Eigenvalue spectrum
    eigenvalues = compute_gram_spectrum(W)
    
    # Power traces
    traces = compute_power_traces(W, max_k=4)
    
    # Key statistics
    n_nonzero = (np.abs(eigenvalues) > 1e-8).sum()
    effective_rank = (eigenvalues.sum() ** 2) / (eigenvalues ** 2).sum() if eigenvalues.sum() > 0 else 0
    
    return {
        'eigenvalues': eigenvalues,
        'traces': traces,
        'n_nonzero_eigenvalues': n_nonzero,
        'effective_rank': effective_rank,
        'spectral_gap': eigenvalues[0] - eigenvalues[1] if len(eigenvalues) > 1 else 0
    }

# Analyze a sample of experiments
sample_keys = list(all_data.keys())[:20]  # First 20
spectral_results = {}

print("Computing spectral analysis for sample experiments...")
for key in tqdm(sample_keys):
    spectral_results[key] = analyze_experiment_spectrum(all_data[key])

# Show one example
example_key = sample_keys[0]
print(f"\nExample: m={example_key[0]}, s={example_key[1]:.3f}")
print(f"  Top 10 eigenvalues: {spectral_results[example_key]['eigenvalues'][:10]}")
print(f"  Tr(M^k): {spectral_results[example_key]['traces']}")
print(f"  Effective rank: {spectral_results[example_key]['effective_rank']:.2f}")

## 11. Eigenvalue Distribution vs Compression/Sparsity

In [ ]:
def plot_eigenvalue_distributions(all_data, n_samples=50):
    """Plot eigenvalue distributions across different regimes."""
    
    # Sample experiments across the parameter space
    keys = list(all_data.keys())
    sample_indices = np.linspace(0, len(keys)-1, min(n_samples, len(keys))).astype(int)
    sample_keys = [keys[i] for i in sample_indices]
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    
    # Collect data
    all_top_eigs = []
    all_rhos = []
    all_sparsities = []
    effective_ranks = []
    
    for key in tqdm(sample_keys, desc='Computing spectra'):
        m, s, seed = key
        W = all_data[key]['weights'][-1]
        eigs = compute_gram_spectrum(W)
        
        rho = N_FEATURES / m
        all_top_eigs.append(eigs[0])  # Largest eigenvalue
        all_rhos.append(rho)
        all_sparsities.append(s)
        
        eff_rank = (eigs.sum() ** 2) / ((eigs ** 2).sum() + 1e-10)
        effective_ranks.append(eff_rank)
    
    all_top_eigs = np.array(all_top_eigs)
    all_rhos = np.array(all_rhos)
    all_sparsities = np.array(all_sparsities)
    effective_ranks = np.array(effective_ranks)
    
    # Top eigenvalue vs compression ratio
    sc1 = axes[0, 0].scatter(all_rhos, all_top_eigs, c=all_sparsities, cmap='viridis', alpha=0.7)
    axes[0, 0].set_xlabel('Compression Ratio $n/m$')
    axes[0, 0].set_ylabel('Largest Eigenvalue $\\mu_1$')
    axes[0, 0].set_title('Top Eigenvalue vs Compression')
    plt.colorbar(sc1, ax=axes[0, 0], label='Sparsity')
    
    # Top eigenvalue vs sparsity
    sc2 = axes[0, 1].scatter(all_sparsities, all_top_eigs, c=all_rhos, cmap='plasma', alpha=0.7)
    axes[0, 1].set_xlabel('Sparsity $S$')
    axes[0, 1].set_ylabel('Largest Eigenvalue $\\mu_1$')
    axes[0, 1].set_title('Top Eigenvalue vs Sparsity')
    plt.colorbar(sc2, ax=axes[0, 1], label='$n/m$')
    
    # Effective rank vs parameters
    sc3 = axes[1, 0].scatter(all_rhos, effective_ranks, c=all_sparsities, cmap='viridis', alpha=0.7)
    axes[1, 0].set_xlabel('Compression Ratio $n/m$')
    axes[1, 0].set_ylabel('Effective Rank')
    axes[1, 0].set_title('Effective Rank vs Compression')
    plt.colorbar(sc3, ax=axes[1, 0], label='Sparsity')
    
    # Effective rank vs sparsity
    sc4 = axes[1, 1].scatter(all_sparsities, effective_ranks, c=all_rhos, cmap='plasma', alpha=0.7)
    axes[1, 1].set_xlabel('Sparsity $S$')
    axes[1, 1].set_ylabel('Effective Rank')
    axes[1, 1].set_title('Effective Rank vs Sparsity')
    plt.colorbar(sc4, ax=axes[1, 1], label='$n/m$')
    
    plt.suptitle('Eigenvalue Statistics of $M = W^\\intercal W$', fontsize=14, weight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'eigenvalue_distributions.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_eigenvalue_distributions(all_data, n_samples=100)

## 12. Primitive Idempotent Decomposition

Testing the ansatz: $M \approx \sum_j \mu_j E_j$ where $E_j$ are primitive idempotents.

In [ ]:
def analyze_idempotent_structure(W, n_top=10, tol=1e-4):
    """
    Analyze how well M approximates a sum of primitive idempotents.
    
    For M = mu E_o (single idempotent), M^2 = mu M.
    Returns the ratio ||M^2 - mu M|| / ||M|| as a measure of deviation.
    """
    M = W.T @ W
    
    # Full eigendecomposition
    eigenvalues, eigenvectors = np.linalg.eigh(M)
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]
    
    # Group eigenvalues by similarity (detecting multiplicities)
    unique_eigs = []
    multiplicities = []
    
    for ev in eigenvalues:
        if np.abs(ev) < tol:  # Skip near-zero eigenvalues
            continue
        found = False
        for i, ue in enumerate(unique_eigs):
            if np.abs(ev - ue) < tol * max(1, np.abs(ue)):
                multiplicities[i] += 1
                found = True
                break
        if not found:
            unique_eigs.append(ev)
            multiplicities.append(1)
    
    # Compute idempotent approximation error for dominant eigenvalue
    if len(unique_eigs) > 0:
        mu = unique_eigs[0]  # Dominant eigenvalue
        M_squared = M @ M
        idempotent_error = np.linalg.norm(M_squared - mu * M, 'fro') / (np.linalg.norm(M, 'fro') + 1e-10)
    else:
        idempotent_error = np.nan
        mu = 0
    
    return {
        'unique_eigenvalues': unique_eigs[:n_top],
        'multiplicities': multiplicities[:n_top],
        'dominant_mu': mu,
        'idempotent_error': idempotent_error,
        'n_distinct_geometries': len([e for e in unique_eigs if np.abs(e) > tol])
    }

# Analyze a representative sample
print("Analyzing idempotent structure...")
idempotent_results = []

sample_keys = list(all_data.keys())[::50]  # Every 50th experiment
for key in tqdm(sample_keys):
    m, s, seed = key
    W = all_data[key]['weights'][-1]
    result = analyze_idempotent_structure(W)
    result['m'] = m
    result['s'] = s
    result['rho'] = N_FEATURES / m
    idempotent_results.append(result)

# Display results
print("\nSample idempotent analysis:")
for r in idempotent_results[:5]:
    print(f"\nm={r['m']}, S={r['s']:.3f}, rho={r['rho']:.2f}:")
    print(f"  Distinct geometries (non-zero eigenvalues): {r['n_distinct_geometries']}")
    print(f"  Top eigenvalues: {r['unique_eigenvalues'][:5]}")
    print(f"  Multiplicities: {r['multiplicities'][:5]}")
    print(f"  Idempotent error ||M^2-muM||/||M||: {r['idempotent_error']:.4f}")

## 13. Resolvent / Stieltjes Transform Analysis

The Stieltjes transform:
$$m(z) = \frac{1}{n}\text{Tr}(M-zI)^{-1} = \sum_{j=1}^k\frac{m_j/n}{\mu_j-z}$$

Poles reveal the number of distinct geometries.

In [ ]:
def compute_resolvent(W, z_values):
    """
    Compute the Stieltjes transform m(z) = (1/n) Tr(M - zI)^{-1}
    for a range of z values along the real axis.
    """
    M = W.T @ W
    n = M.shape[0]
    
    eigenvalues = np.linalg.eigvalsh(M)
    
    # Compute resolvent for each z
    # m(z) = (1/n) * sum_i 1/(lambda_i - z)
    resolvent = np.zeros(len(z_values), dtype=complex)
    
    for i, z in enumerate(z_values):
        resolvent[i] = np.mean(1.0 / (eigenvalues - z))
    
    return resolvent

def plot_resolvent_analysis(all_data, m_target=256, s_target=0.5, seed=0):
    """Plot the resolvent to identify poles (distinct geometries)."""
    
    # Find matching experiment
    best_key = None
    best_dist = float('inf')
    for (m, s, sd) in all_data.keys():
        if sd == seed:
            dist = abs(m - m_target) / 100 + abs(s - s_target)
            if dist < best_dist:
                best_dist = dist
                best_key = (m, s, sd)
    
    if best_key is None:
        print("No matching experiment found")
        return
    
    data = all_data[best_key]
    W = data['weights'][-1]
    M = W.T @ W
    
    eigenvalues = np.linalg.eigvalsh(M)
    eigenvalues = np.sort(eigenvalues)[::-1]
    
    # Compute resolvent along real axis (with small imaginary part for regularization)
    z_real = np.linspace(-0.5, eigenvalues[0] * 1.5, 500)
    epsilon = 0.01 * eigenvalues[0]  # Small imaginary part
    z_values = z_real + 1j * epsilon
    
    resolvent = compute_resolvent(W, z_values)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Eigenvalue histogram
    axes[0, 0].hist(eigenvalues[eigenvalues > 1e-6], bins=50, density=True, 
                    color='steelblue', alpha=0.7, edgecolor='white')
    axes[0, 0].set_xlabel('Eigenvalue $\\mu$')
    axes[0, 0].set_ylabel('Density')
    axes[0, 0].set_title('Eigenvalue Distribution of $M$')
    
    # |m(z)| - poles show as peaks
    axes[0, 1].semilogy(z_real, np.abs(resolvent), 'b-', linewidth=1)
    axes[0, 1].set_xlabel('$z$ (real part)')
    axes[0, 1].set_ylabel('$|m(z)|$')
    axes[0, 1].set_title('Stieltjes Transform Magnitude')
    
    # Mark eigenvalue locations
    for ev in eigenvalues[:10]:
        if ev > 1e-6:
            axes[0, 1].axvline(ev, color='red', alpha=0.3, linestyle='--')
    
    # Im(m(z)) - peaks indicate eigenvalue locations
    axes[1, 0].plot(z_real, np.imag(resolvent), 'g-', linewidth=1)
    axes[1, 0].set_xlabel('$z$ (real part)')
    axes[1, 0].set_ylabel('Im$(m(z))$')
    axes[1, 0].set_title('Imaginary Part of Resolvent')
    
    # Spectral density: rho(mu) = -(1/pi) Im(m(mu + i*epsilon))
    spectral_density = -np.imag(resolvent) / np.pi
    axes[1, 1].plot(z_real, spectral_density, 'purple', linewidth=1)
    axes[1, 1].fill_between(z_real, spectral_density, alpha=0.3, color='purple')
    axes[1, 1].set_xlabel('$\\mu$')
    axes[1, 1].set_ylabel('$\\rho(\\mu)$')
    axes[1, 1].set_title('Spectral Density from Resolvent')
    axes[1, 1].set_xlim(0, eigenvalues[0] * 1.2)
    
    plt.suptitle(f'Resolvent Analysis: m={best_key[0]}, S={best_key[1]:.3f}', fontsize=14, weight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'resolvent_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Count distinct poles (local maxima in spectral density)
    from scipy.signal import find_peaks
    peaks, _ = find_peaks(spectral_density, height=0.1*spectral_density.max(), distance=10)
    print(f"\nDetected {len(peaks)} distinct spectral peaks (potential geometries)")
    print(f"Peak locations: {z_real[peaks]}")

plot_resolvent_analysis(all_data, m_target=256, s_target=0.5)
plot_resolvent_analysis(all_data, m_target=128, s_target=0.8)

## 14. Testing the Conservation Law: $\sum_i D_i = m_0$ (Multiplicity)

In [ ]:
def test_conservation_law(all_data):
    """
    Test if sum(D_i) relates to eigenspace multiplicity.
    
    From the theory:
    D_i = ||W_i||^2 / mu
    sum_i D_i = Tr(M) / mu = mu * m_0 / mu = m_0
    
    where m_0 is the multiplicity of the dominant eigenspace.
    """
    
    results = []
    
    for (m, s, seed), data in tqdm(list(all_data.items())[::20], desc='Testing conservation'):
        W = data['weights'][-1]
        M = W.T @ W
        
        # Get D_i from data
        D_i = data['fractional_dims'][-1]
        sum_D_i = D_i.sum()
        
        # Compute eigenvalues
        eigenvalues = np.linalg.eigvalsh(M)
        eigenvalues = np.sort(eigenvalues)[::-1]
        
        # Dominant eigenvalue and its multiplicity
        mu_dominant = eigenvalues[0]
        tol = 0.1 * mu_dominant
        multiplicity = np.sum(np.abs(eigenvalues - mu_dominant) < tol)
        
        # Trace-based estimate: Tr(M) / mu
        trace_M = np.trace(M)
        trace_estimate = trace_M / (mu_dominant + 1e-10)
        
        results.append({
            'm': m,
            's': s,
            'rho': N_FEATURES / m,
            'sum_D_i': sum_D_i,
            'multiplicity': multiplicity,
            'trace_estimate': trace_estimate,
            'mu_dominant': mu_dominant
        })
    
    return results

conservation_results = test_conservation_law(all_data)

# Plot results
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

sum_D = [r['sum_D_i'] for r in conservation_results]
mult = [r['multiplicity'] for r in conservation_results]
trace_est = [r['trace_estimate'] for r in conservation_results]
rhos = [r['rho'] for r in conservation_results]
ms = [r['m'] for r in conservation_results]

# sum(D_i) vs m_hidden
sc1 = axes[0].scatter(ms, sum_D, c=[r['s'] for r in conservation_results], cmap='viridis', alpha=0.7)
axes[0].plot([min(ms), max(ms)], [min(ms), max(ms)], 'k--', label='sum(D_i) = m')
axes[0].set_xlabel('$m$ (hidden dimension)')
axes[0].set_ylabel('$\\sum_i D_i$')
axes[0].set_title('Conservation Law Test')
axes[0].legend()
plt.colorbar(sc1, ax=axes[0], label='Sparsity')

# sum(D_i) vs Tr(M)/mu
axes[1].scatter(trace_est, sum_D, c=rhos, cmap='plasma', alpha=0.7)
lim = max(max(trace_est), max(sum_D))
axes[1].plot([0, lim], [0, lim], 'k--')
axes[1].set_xlabel('$\\text{Tr}(M)/\\mu$')
axes[1].set_ylabel('$\\sum_i D_i$')
axes[1].set_title('Sum of $D_i$ vs Trace Estimate')

# Ratio analysis
ratios = [r['sum_D_i'] / r['m'] for r in conservation_results]
axes[2].hist(ratios, bins=30, color='steelblue', alpha=0.7, edgecolor='white')
axes[2].axvline(1.0, color='red', linestyle='--', label='sum(D_i)/m = 1')
axes[2].set_xlabel('$\\sum_i D_i / m$')
axes[2].set_ylabel('Count')
axes[2].set_title('Ratio Distribution')
axes[2].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'conservation_law.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nMean ratio sum(D_i)/m: {np.mean(ratios):.4f} +/- {np.std(ratios):.4f}")

## 15. Spectral Ray Quantization: Testing $D_i = \|W_i\|^2 / \mu$

In [ ]:
def analyze_ray_quantization(all_data, n_samples=30):
    """
    Analyze how well the spectral ansatz D_i = ||W_i||^2 / mu holds.
    
    If features belong to a single geometry with eigenvalue mu,
    the ratio D_i * mu / ||W_i||^2 should equal 1.
    """
    
    sample_keys = list(all_data.keys())[::len(all_data)//n_samples][:n_samples]
    
    all_predicted = []  # D_i predicted from ||W||^2/mu
    all_actual = []     # Actual D_i
    all_colors = []     # For coloring
    
    for key in tqdm(sample_keys, desc='Analyzing quantization'):
        m, s, seed = key
        data = all_data[key]
        
        W = data['weights'][-1]
        M = W.T @ W
        
        # Get dominant eigenvalue
        eigenvalues = np.linalg.eigvalsh(M)
        mu = np.max(eigenvalues)
        
        # Actual D_i and norms
        D_i = data['fractional_dims'][-1]
        norms = data['feature_norms'][-1]
        
        # Predicted D_i under single-geometry ansatz
        D_i_pred = norms / (mu + 1e-10)
        
        # Filter active features
        mask = norms > 0.01
        
        all_predicted.extend(D_i_pred[mask].tolist())
        all_actual.extend(D_i[mask].tolist())
        all_colors.extend([s] * mask.sum())
    
    all_predicted = np.array(all_predicted)
    all_actual = np.array(all_actual)
    all_colors = np.array(all_colors)
    
    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Predicted vs Actual
    sc = axes[0].scatter(all_predicted, all_actual, c=all_colors, cmap='viridis', 
                         s=2, alpha=0.3, rasterized=True)
    axes[0].plot([0, 1], [0, 1], 'r--', linewidth=2, label='Perfect prediction')
    axes[0].set_xlabel('Predicted $D_i = \|W_i\|^2 / \\mu$')
    axes[0].set_ylabel('Actual $D_i$')
    axes[0].set_title('Spectral Ansatz Validation')
    axes[0].legend()
    axes[0].set_xlim(0, 1.1)
    axes[0].set_ylim(0, 1.1)
    plt.colorbar(sc, ax=axes[0], label='Sparsity')
    
    # Residuals histogram
    residuals = all_actual - all_predicted
    axes[1].hist(residuals, bins=100, density=True, color='steelblue', alpha=0.7, edgecolor='white')
    axes[1].axvline(0, color='red', linestyle='--', label=f'mu={np.mean(residuals):.4f}')
    axes[1].set_xlabel('$D_i^{actual} - D_i^{predicted}$')
    axes[1].set_ylabel('Density')
    axes[1].set_title('Residual Distribution')
    axes[1].legend()
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'ray_quantization.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Compute R^2
    ss_res = np.sum(residuals**2)
    ss_tot = np.sum((all_actual - all_actual.mean())**2)
    r_squared = 1 - ss_res / ss_tot
    print(f"\nR^2 score for spectral ansatz: {r_squared:.4f}")
    print(f"Mean residual: {np.mean(residuals):.4f}")
    print(f"Std residual: {np.std(residuals):.4f}")

analyze_ray_quantization(all_data)

## 16. Power Trace Evolution: Newton's Identities

In [ ]:
def plot_power_traces_evolution(all_data, m_target=256, s_target=0.5, seed=0):
    """
    Track Tr(M^k) for k=1,2,3,4 during training.
    
    These power sums determine elementary symmetric polynomials
    via Newton's identities, sufficient for identifying standard geometries.
    """
    
    # Find matching experiment
    best_key = None
    best_dist = float('inf')
    for (m, s, sd) in all_data.keys():
        if sd == seed:
            dist = abs(m - m_target) / 100 + abs(s - s_target)
            if dist < best_dist:
                best_dist = dist
                best_key = (m, s, sd)
    
    if best_key is None:
        print("No matching experiment found")
        return
    
    data = all_data[best_key]
    print(f"Analyzing: m={best_key[0]}, s={best_key[1]:.3f}")
    
    steps = data['checkpoint_steps']
    weights = data['weights']  # (n_checkpoints, m, n)
    
    # Compute traces at each checkpoint
    trace_k = {1: [], 2: [], 3: [], 4: []}
    
    for i in range(len(steps)):
        W = weights[i]
        traces = compute_power_traces(W, max_k=4)
        for k in range(1, 5):
            trace_k[k].append(traces[k-1])
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    for k, ax in zip(range(1, 5), axes.flat):
        ax.plot(steps, trace_k[k], 'b-', linewidth=2)
        ax.set_xlabel('Training Step')
        ax.set_ylabel(f'$\\text{{Tr}}(M^{k})$')
        ax.set_title(f'Power Trace k={k}')
        ax.grid(True, alpha=0.3)
    
    plt.suptitle(f'Power Trace Evolution: m={best_key[0]}, S={best_key[1]:.3f}', fontsize=14, weight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'power_traces.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Normalized traces (invariants)
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Plot ratios that should converge to characteristic values for different geometries
    tr1 = np.array(trace_k[1])
    tr2 = np.array(trace_k[2])
    tr3 = np.array(trace_k[3])
    tr4 = np.array(trace_k[4])
    
    # Ratio Tr(M^2)/Tr(M)^2 indicates concentration of eigenvalues
    ratio_2 = tr2 / (tr1**2 + 1e-10)
    ratio_3 = tr3 / (tr1**3 + 1e-10)
    ratio_4 = tr4 / (tr1**4 + 1e-10)
    
    ax.plot(steps, ratio_2, 'r-', linewidth=2, label='$\\text{Tr}(M^2)/\\text{Tr}(M)^2$')
    ax.plot(steps, ratio_3, 'g-', linewidth=2, label='$\\text{Tr}(M^3)/\\text{Tr}(M)^3$')
    ax.plot(steps, ratio_4, 'b-', linewidth=2, label='$\\text{Tr}(M^4)/\\text{Tr}(M)^4$')
    
    ax.set_xlabel('Training Step')
    ax.set_ylabel('Normalized Trace Ratio')
    ax.set_title('Geometry Invariants from Newton\'s Identities')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'newton_invariants.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_power_traces_evolution(all_data, m_target=256, s_target=0.5)

## 17. Multi-Geometry Phase Detection

In [ ]:
def detect_geometry_phases(all_data, eigenvalue_gap_threshold=0.1):
    """
    Detect how many distinct spectral phases (geometries) emerge
    as a function of compression and sparsity.
    
    Uses spectral gap analysis to count significant eigenvalue clusters.
    """
    
    results = []
    
    for (m, s, seed), data in tqdm(list(all_data.items())[::10], desc='Detecting phases'):
        W = data['weights'][-1]
        eigenvalues = compute_gram_spectrum(W)
        
        # Filter significant eigenvalues
        sig_eigs = eigenvalues[eigenvalues > eigenvalue_gap_threshold * eigenvalues[0]]
        
        if len(sig_eigs) < 2:
            n_clusters = 1
        else:
            # Cluster eigenvalues by relative gap
            try:
                eig_diffs = np.diff(sig_eigs)
                rel_gaps = np.abs(eig_diffs) / (np.abs(sig_eigs[:-1]) + 1e-10)
                n_clusters = 1 + np.sum(rel_gaps > 0.3)  # Count significant gaps
            except:
                n_clusters = 1
        
        results.append({
            'm': m,
            's': s,
            'rho': N_FEATURES / m,
            'n_geometries': min(n_clusters, 10),  # Cap at 10
            'spectral_entropy': -np.sum((sig_eigs/sig_eigs.sum()) * np.log(sig_eigs/sig_eigs.sum() + 1e-10))
        })
    
    return results

phase_results = detect_geometry_phases(all_data)

# Create heatmap
m_unique = sorted(set(r['m'] for r in phase_results))
s_unique = sorted(set(r['s'] for r in phase_results))

geometry_grid = np.full((len(m_unique), len(s_unique)), np.nan)
entropy_grid = np.full((len(m_unique), len(s_unique)), np.nan)

for r in phase_results:
    i = m_unique.index(r['m'])
    j = s_unique.index(r['s']) if r['s'] in s_unique else -1
    if j >= 0:
        geometry_grid[i, j] = r['n_geometries']
        entropy_grid[i, j] = r['spectral_entropy']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

im1 = ax1.imshow(geometry_grid, aspect='auto', cmap='YlOrRd', origin='lower')
ax1.set_xlabel('Sparsity Index')
ax1.set_ylabel('m_hidden Index')
ax1.set_title('Number of Distinct Geometries (Spectral Clusters)')
plt.colorbar(im1, ax=ax1, label='# Geometries')

im2 = ax2.imshow(entropy_grid, aspect='auto', cmap='viridis', origin='lower')
ax2.set_xlabel('Sparsity Index')
ax2.set_ylabel('m_hidden Index')
ax2.set_title('Spectral Entropy')
plt.colorbar(im2, ax=ax2, label='Entropy')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'geometry_phases.png', dpi=150, bbox_inches='tight')
plt.show()

## 18. Summary Statistics

In [ ]:
print("=" * 60)
print("SPECTRAL SUPERPOSITION ANALYSIS SUMMARY")
print("=" * 60)

print(f"\nDataset: {len(all_data)} valid experiments")
print(f"  m_hidden range: {min(m_values)} to {max(m_values)} ({len(m_values)} values)")
print(f"  Sparsity range: {min(s_values):.3f} to {max(s_values):.3f} ({len(s_values)} values)")
print(f"  Seeds: {seeds}")
print(f"  Features per experiment: {N_FEATURES}")
print(f"  Checkpoints per experiment: 56")

if conservation_results:
    ratios = [r['sum_D_i'] / r['m'] for r in conservation_results]
    print(f"\nConservation Law (sum(D_i)/m):")
    print(f"  Mean: {np.mean(ratios):.4f} (expected: 1.0)")
    print(f"  Std:  {np.std(ratios):.4f}")

if phase_results:
    n_geoms = [r['n_geometries'] for r in phase_results]
    print(f"\nGeometry Detection:")
    print(f"  Mean distinct geometries: {np.mean(n_geoms):.2f}")
    print(f"  Max distinct geometries: {np.max(n_geoms)}")

print(f"\nOutput files saved to: {OUTPUT_DIR.absolute()}")